# Week 1 Baseline — RED-CNN on Mayo + Cross-Vendor Test on Piglet

**Goal:** Reproduce RED-CNN (Chen 2017, IEEE TMI) on Mayo, then evaluate on Piglet (GE scanner).
The Mayo→Piglet PSNR gap is the motivation experiment for the TTA method developed in subsequent weeks.

**Success criteria:**
- Mayo validation PSNR ≥ 35 dB at 25% dose
- Measurable Mayo→Piglet PSNR drop (expect 2-5 dB) — this is the paper's hook

**Datasets:**
- Mayo (Siemens): `/media/ant-pc/HDD/datasets/ct-low-dose-reconstruction/Preprocessed_512x512/`
- Piglet (GE):    `/media/ant-pc/HDD/datasets/piglet/piglet/DICOM/PA0/ST0/`

**Honest disclaimer:** This is a STARTER notebook. Paths/formats may need adjustment depending on the exact Mayo Preprocessed structure. The inspection cells (Section 1) will reveal what's actually there — adjust the dataset class accordingly if needed.


## 0. Setup


In [ ]:
import os, glob, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import pydicom

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
CONFIG = {
    # Paths
    'mayo_root':   '/media/ant-pc/HDD/datasets/ct-low-dose-reconstruction/Preprocessed_512x512',
    'piglet_root': '/media/ant-pc/HDD/datasets/piglet/piglet/DICOM/PA0/ST0',
    'output_dir':  '/media/ant-pc/HDD/experiments/week1_baseline',

    # Training
    'batch_size':         16,
    'patch_size':         64,
    'patches_per_image':  10,
    'lr':                 1e-4,
    'epochs':             50,
    'val_every':          5,
    'num_workers':        4,

    # LDCT simulation (used only if Mayo Preprocessed has NDCT-only)
    'dose_factor':  0.25,   # 25% dose
    'I0':           1.0e5,  # incident photon count at full dose

    # HU normalization
    'hu_min': -1024.0,
    'hu_max':  3071.0,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 1. Data Inspection

Before building loaders, inspect what's actually in the Mayo Preprocessed folder and the Piglet DICOM tree. The Mayo Preprocessed format depends on however it was preprocessed — could be paired LDCT/NDCT `.npy` files, or NDCT-only (in which case we simulate LDCT).


In [ ]:
def inspect_mayo(root):
    """Walk Mayo root, report file types and structure."""
    root = Path(root)
    if not root.exists():
        print(f'❌ Path does not exist: {root}')
        return None

    ext_counts = {}
    sample_paths = []
    for p in root.rglob('*'):
        if p.is_file():
            ext = p.suffix.lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
            if len(sample_paths) < 8:
                sample_paths.append(str(p.relative_to(root)))

    print(f'Mayo root: {root}')
    print(f'File extensions found: {ext_counts}')
    print(f'Sample relative paths:')
    for s in sample_paths:
        print(f'  {s}')

    # Probe .npy files
    npy_files = sorted(root.rglob('*.npy'))
    if npy_files:
        print(f'\n.npy files: {len(npy_files)}')
        print(f'First 3 with shape/dtype/range:')
        for f in npy_files[:3]:
            arr = np.load(f, mmap_mode='r')
            print(f'  {f.relative_to(root)}: shape={arr.shape}, dtype={arr.dtype}, range=[{float(arr.min()):.2f}, {float(arr.max()):.2f}]')
    else:
        print('\n⚠️  No .npy files found. Check structure manually.')

    # Probe HDF5
    h5_files = sorted(root.rglob('*.h5')) + sorted(root.rglob('*.hdf5'))
    if h5_files:
        print(f'\n.h5 files: {len(h5_files)}')

    return ext_counts, npy_files

mayo_inspection = inspect_mayo(CONFIG['mayo_root'])


In [ ]:
def inspect_piglet(root):
    """Build Piglet series metadata table."""
    root = Path(root)
    if not root.exists():
        print(f'❌ Path does not exist: {root}')
        return None

    series_dirs = sorted(
        [d for d in root.iterdir() if d.is_dir() and d.name.startswith('SE')],
        key=lambda x: int(x.name[2:])
    )

    info = []
    for sd in series_dirs:
        files = sorted(sd.iterdir())
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[0]), stop_before_pixels=True)
            info.append({
                'series':           sd.name,
                'n_slices':         len(files),
                'mAs':              float(ds.get('Exposure', 0)),
                'kVp':              float(ds.get('KVP', 0)),
                'kernel':           str(ds.get('ConvolutionKernel', '?')),
                'slice_thickness':  float(ds.get('SliceThickness', 0)),
                'manufacturer':     str(ds.get('Manufacturer', '?')),
            })
        except Exception as e:
            print(f'  Skip {sd.name}: {e}')

    return info

piglet_info = inspect_piglet(CONFIG['piglet_root'])
print(f'Piglet series found: {len(piglet_info)}')
print()
print(f"{'Series':<8} {'Slices':<8} {'mAs':<6} {'kVp':<6} {'Kernel':<14} {'Thick':<6}")
print('-' * 55)
for s in piglet_info:
    print(f"{s['series']:<8} {s['n_slices']:<8} {s['mAs']:<6.0f} {s['kVp']:<6.0f} {s['kernel']:<14} {s['slice_thickness']:<6.1f}")


In [ ]:
# Pick Piglet pairs: highest mAs as reference, mid-range mAs as low-dose test
piglet_sorted = sorted(piglet_info, key=lambda x: -x['mAs'])
PIGLET_REF = piglet_sorted[0]['series']

low_candidates = [s['series'] for s in piglet_sorted if s['mAs'] < piglet_sorted[0]['mAs'] and s['mAs'] > 0]
PIGLET_LOW = low_candidates[len(low_candidates) // 2] if low_candidates else (low_candidates[0] if low_candidates else None)

print(f'Reference (high-mAs): {PIGLET_REF}  ({piglet_sorted[0]["mAs"]:.0f} mAs)')
print(f'Low-dose (test input): {PIGLET_LOW}')

# Save mapping for downstream
with open(os.path.join(CONFIG['output_dir'], 'piglet_series.json'), 'w') as f:
    json.dump({'ref': PIGLET_REF, 'low': PIGLET_LOW, 'all': piglet_info}, f, indent=2)


## 2. Datasets

**Mayo dataset:** Auto-detects paired vs NDCT-only structure. If paired LDCT/NDCT files exist (heuristics: subdirs `LDCT/`+`NDCT/`, or filename markers like `_ldct`/`_ndct`), use them directly. Otherwise treat all files as NDCT and simulate LDCT via image-domain Poisson-Gaussian approximation.

**Piglet dataset:** DICOM loader. Matches slices across series by `InstanceNumber`. HU conversion via `RescaleSlope`/`RescaleIntercept`.


In [ ]:
def normalize_hu(img, hu_min, hu_max):
    """Map HU values to [0, 1]."""
    img = (img.astype(np.float32) - hu_min) / (hu_max - hu_min)
    return np.clip(img, 0.0, 1.0)


def simulate_ldct_image_domain(ndct, dose_factor=0.25, I0=1e5):
    """
    Image-domain Poisson-Gaussian approximation for fast LDCT simulation.

    For Week 1 baseline this is acceptable. For final paper, replace with
    full sinogram-domain Zeng 2015 simulation using torch-radon or similar.

    Args:
        ndct:        clean image in [0, 1]
        dose_factor: relative dose, e.g. 0.25 for 25%
        I0:          incident photon count at full dose
    """
    I0_eff = I0 * dose_factor
    sigma = np.sqrt(np.maximum(ndct, 0.01) / I0_eff)
    noise = np.random.normal(0.0, sigma, size=ndct.shape).astype(np.float32)
    return np.clip(ndct + noise, 0.0, 1.0)


In [ ]:
class MayoDataset(Dataset):
    """
    Mayo dataset with auto-detection of paired vs NDCT-only structure.
    """

    def __init__(self, root, patch_size=64, patches_per_image=10,
                 mode='train', dose_factor=0.25, I0=1e5,
                 hu_min=-1024, hu_max=3071):
        self.root              = Path(root)
        self.patch_size        = patch_size
        self.patches_per_image = patches_per_image
        self.mode              = mode
        self.dose_factor       = dose_factor
        self.I0                = I0
        self.hu_min            = hu_min
        self.hu_max            = hu_max

        self.pairs, self.simulated = self._discover_pairs()
        if not self.pairs:
            raise RuntimeError(
                f'No images found under {root}. Inspect manually and adjust.'
            )
        print(f'Mayo [{mode}]: {len(self.pairs)} image pairs, simulated_ldct={self.simulated}')

    def _discover_pairs(self):
        npy_files = sorted(self.root.rglob('*.npy'))
        if not npy_files:
            return [], False

        # Heuristic 1: dedicated subdirs
        for ld_name, nd_name in [('LDCT', 'NDCT'), ('ldct', 'ndct'),
                                  ('low_dose', 'normal_dose'), ('input', 'target')]:
            ld_dir = self.root / ld_name
            nd_dir = self.root / nd_name
            if ld_dir.exists() and nd_dir.exists():
                ld_files = sorted(ld_dir.rglob('*.npy'))
                nd_files = sorted(nd_dir.rglob('*.npy'))
                if ld_files and len(ld_files) == len(nd_files):
                    return list(zip(ld_files, nd_files)), False

        # Heuristic 2: filename markers
        ld_files = [f for f in npy_files
                    if any(tok in f.name.lower() for tok in ['ldct', '_ld_', 'low_dose', 'quarter'])]
        nd_files = [f for f in npy_files
                    if any(tok in f.name.lower() for tok in ['ndct', '_nd_', 'full_dose', 'normal'])]
        if ld_files and nd_files and len(ld_files) == len(nd_files):
            return list(zip(sorted(ld_files), sorted(nd_files))), False

        # Fallback: NDCT-only, simulate
        return [(None, f) for f in npy_files], True

    def _load_npy(self, path):
        arr = np.load(path)
        if arr.ndim == 3:
            # If volume, take middle slice (Week 1 simplicity)
            arr = arr[arr.shape[0] // 2]
        if arr.dtype != np.float32:
            arr = arr.astype(np.float32)
        # If still in HU-like range, normalize
        if arr.min() < -10 or arr.max() > 10:
            arr = normalize_hu(arr, self.hu_min, self.hu_max)
        else:
            arr = np.clip(arr, 0.0, 1.0)
        return arr

    def __len__(self):
        if self.mode == 'train':
            return len(self.pairs) * self.patches_per_image
        return len(self.pairs)

    def __getitem__(self, idx):
        pair_idx = idx % len(self.pairs)
        ldct_path, ndct_path = self.pairs[pair_idx]

        ndct = self._load_npy(ndct_path)
        if self.simulated:
            ldct = simulate_ldct_image_domain(ndct, self.dose_factor, self.I0)
        else:
            ldct = self._load_npy(ldct_path)

        if self.mode == 'train':
            h, w = ndct.shape
            ph = self.patch_size
            if h <= ph or w <= ph:
                # Skip cropping for tiny images
                pass
            else:
                top  = np.random.randint(0, h - ph)
                left = np.random.randint(0, w - ph)
                ndct = ndct[top:top+ph, left:left+ph]
                ldct = ldct[top:top+ph, left:left+ph]

        return (
            torch.from_numpy(ldct).unsqueeze(0).float(),
            torch.from_numpy(ndct).unsqueeze(0).float(),
        )


In [ ]:
class PigletDataset(Dataset):
    """
    Piglet cross-vendor test set. Pairs slices across two series by InstanceNumber.
    """

    def __init__(self, root, ref_series, low_series,
                 hu_min=-1024, hu_max=3071, max_slices=300):
        self.root   = Path(root)
        self.hu_min = hu_min
        self.hu_max = hu_max

        ref_list = self._load_series_files(ref_series)
        low_list = self._load_series_files(low_series)

        # Match by InstanceNumber
        ref_map = {inst: p for inst, p in ref_list}
        low_map = {inst: p for inst, p in low_list}
        common  = sorted(set(ref_map) & set(low_map))
        common  = common[:max_slices]

        self.ref_files = [ref_map[i] for i in common]
        self.low_files = [low_map[i] for i in common]
        print(f'Piglet: ref={ref_series}, low={low_series}, matched slices={len(common)}')

    def _load_series_files(self, series_name):
        series_dir = self.root / series_name
        out = []
        for f in series_dir.iterdir():
            try:
                ds = pydicom.dcmread(str(f), stop_before_pixels=True)
                out.append((int(ds.InstanceNumber), str(f)))
            except Exception:
                continue
        out.sort()
        return out

    def _load_dicom(self, path):
        ds        = pydicom.dcmread(path)
        img       = ds.pixel_array.astype(np.float32)
        slope     = float(ds.get('RescaleSlope', 1.0))
        intercept = float(ds.get('RescaleIntercept', 0.0))
        hu        = img * slope + intercept
        return normalize_hu(hu, self.hu_min, self.hu_max)

    def __len__(self):
        return len(self.ref_files)

    def __getitem__(self, idx):
        low = self._load_dicom(self.low_files[idx])
        ref = self._load_dicom(self.ref_files[idx])
        return (
            torch.from_numpy(low).unsqueeze(0).float(),
            torch.from_numpy(ref).unsqueeze(0).float(),
        )


In [ ]:
# Sanity check: visualize one Mayo pair and one Piglet pair
mayo_probe = MayoDataset(
    CONFIG['mayo_root'],
    patch_size=CONFIG['patch_size'],
    patches_per_image=1,
    mode='val',
    dose_factor=CONFIG['dose_factor'],
    I0=CONFIG['I0'],
    hu_min=CONFIG['hu_min'],
    hu_max=CONFIG['hu_max'],
)

piglet_probe = PigletDataset(
    CONFIG['piglet_root'], PIGLET_REF, PIGLET_LOW,
    hu_min=CONFIG['hu_min'], hu_max=CONFIG['hu_max'], max_slices=300,
)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

ldct, ndct = mayo_probe[0]
axes[0, 0].imshow(ldct[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title(f'Mayo LDCT (sim={mayo_probe.simulated})')
axes[0, 1].imshow(ndct[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Mayo NDCT')
axes[0, 2].imshow((ndct - ldct)[0].numpy(), cmap='RdBu', vmin=-0.2, vmax=0.2)
axes[0, 2].set_title('NDCT − LDCT')

low, ref = piglet_probe[len(piglet_probe) // 2]
axes[1, 0].imshow(low[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title(f'Piglet {PIGLET_LOW} (low mAs)')
axes[1, 1].imshow(ref[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title(f'Piglet {PIGLET_REF} (high mAs)')
axes[1, 2].imshow((ref - low)[0].numpy(), cmap='RdBu', vmin=-0.2, vmax=0.2)
axes[1, 2].set_title('Ref − Low')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'sample_check.png'), dpi=100)
plt.show()


## 3. RED-CNN Model

Chen et al. 2017, IEEE TMI 36(12):2524-2535. Residual encoder-decoder with 5 conv + 5 deconv layers, 96 filters, 5×5 kernels, symmetric shortcut connections. ~1.85M parameters.


In [ ]:
class REDCNN(nn.Module):
    """
    Residual Encoder-Decoder CNN for LDCT denoising.
    Chen et al. 2017, IEEE TMI 36(12):2524-2535.
    """

    def __init__(self, in_ch=1, out_ch=1, n_filters=96):
        super().__init__()
        kw = dict(kernel_size=5, stride=1, padding=0)

        self.conv1 = nn.Conv2d(in_ch,     n_filters, **kw)
        self.conv2 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv3 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv4 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv5 = nn.Conv2d(n_filters, n_filters, **kw)

        self.deconv1 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv2 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv3 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv4 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv5 = nn.ConvTranspose2d(n_filters, out_ch,    **kw)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        c1 = self.relu(self.conv1(x))
        c2 = self.relu(self.conv2(c1))
        c3 = self.relu(self.conv3(c2))
        c4 = self.relu(self.conv4(c3))
        c5 = self.relu(self.conv5(c4))

        d1 = self.relu(self.deconv1(c5) + c4)
        d2 = self.relu(self.deconv2(d1))
        d3 = self.relu(self.deconv3(d2) + c2)
        d4 = self.relu(self.deconv4(d3))
        d5 = self.deconv5(d4)
        return self.relu(d5 + residual)


model = REDCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'RED-CNN parameters: {n_params:,}  ({n_params/1e6:.2f}M)')

# Forward-pass shape check
with torch.no_grad():
    x = torch.randn(2, 1, 64, 64).to(device)
    y = model(x)
print(f'Input: {tuple(x.shape)}  →  Output: {tuple(y.shape)}')
assert y.shape == x.shape, 'Shape mismatch — check architecture'


## 4. Metrics + Training Loop

L1 loss (more robust than MSE for CT denoising). AdamW + cosine LR. Validate every N epochs, save best by Mayo validation PSNR.


In [ ]:
from skimage.metrics import structural_similarity as ssim_skimage


def compute_psnr(pred, target, data_range=1.0):
    mse = F.mse_loss(pred, target).item()
    if mse <= 0:
        return float('inf')
    return 10.0 * np.log10((data_range ** 2) / mse)


def compute_ssim(pred, target, data_range=1.0):
    p = pred.detach().cpu().numpy().squeeze()
    t = target.detach().cpu().numpy().squeeze()
    if p.ndim == 3:  # batch
        return float(np.mean([ssim_skimage(p[i], t[i], data_range=data_range)
                              for i in range(p.shape[0])]))
    return float(ssim_skimage(p, t, data_range=data_range))


@torch.no_grad()
def evaluate(model, loader, device, max_batches=None):
    model.eval()
    psnrs, ssims = [], []
    for i, (ldct, ndct) in enumerate(tqdm(loader, desc='Eval', leave=False)):
        if max_batches is not None and i >= max_batches:
            break
        ldct = ldct.to(device)
        ndct = ndct.to(device)
        pred = model(ldct)
        for b in range(pred.shape[0]):
            psnrs.append(compute_psnr(pred[b:b+1], ndct[b:b+1]))
            ssims.append(compute_ssim(pred[b:b+1], ndct[b:b+1]))
    return float(np.mean(psnrs)), float(np.mean(ssims))


In [ ]:
def train_model(model, train_loader, val_loader, *, epochs, lr, device, save_path, val_every=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.L1Loss()

    history   = {'train_loss': [], 'val_psnr': [], 'val_ssim': []}
    best_psnr = -1.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_batches  = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for ldct, ndct in pbar:
            ldct = ldct.to(device, non_blocking=True)
            ndct = ndct.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(ldct)
            loss = criterion(pred, ndct)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        scheduler.step()

        avg_loss = epoch_loss / max(1, n_batches)
        history['train_loss'].append(avg_loss)

        do_val = (epoch + 1) % val_every == 0 or epoch == epochs - 1
        if do_val:
            val_psnr, val_ssim = evaluate(model, val_loader, device, max_batches=30)
            history['val_psnr'].append([epoch + 1, val_psnr])
            history['val_ssim'].append([epoch + 1, val_ssim])
            print(f'Epoch {epoch+1:3d}  loss={avg_loss:.4f}  val PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}')
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                torch.save({'model': model.state_dict(),
                            'epoch': epoch + 1,
                            'psnr':  val_psnr,
                            'ssim':  val_ssim}, save_path)

    print(f'\nBest val PSNR: {best_psnr:.2f} dB → {save_path}')
    return history


### Build Mayo loaders and train


In [ ]:
# Build full Mayo dataset, then split image-level (no patient leakage at slice level here;
# for production paper, split at patient level).

mayo_full = MayoDataset(
    CONFIG['mayo_root'],
    patch_size=CONFIG['patch_size'],
    patches_per_image=CONFIG['patches_per_image'],
    mode='train',
    dose_factor=CONFIG['dose_factor'],
    I0=CONFIG['I0'],
    hu_min=CONFIG['hu_min'],
    hu_max=CONFIG['hu_max'],
)

n_images = len(mayo_full.pairs)
n_val    = max(2, n_images // 10)
n_train  = n_images - n_val
all_pairs = mayo_full.pairs

rng = np.random.RandomState(42)
perm = rng.permutation(n_images)
train_pair_indices = sorted(perm[:n_train].tolist())
val_pair_indices   = sorted(perm[n_train:].tolist())
print(f'Mayo: {n_train} train images, {n_val} val images')


class IndexedMayo(Dataset):
    def __init__(self, base, pair_indices, mode):
        self.base   = base
        self.idx    = list(pair_indices)
        self.mode   = mode
    def __len__(self):
        if self.mode == 'train':
            return len(self.idx) * self.base.patches_per_image
        return len(self.idx)
    def __getitem__(self, idx):
        pair_idx = self.idx[idx % len(self.idx)]
        ldct_path, ndct_path = self.base.pairs[pair_idx]
        ndct = self.base._load_npy(ndct_path)
        if self.base.simulated:
            ldct = simulate_ldct_image_domain(ndct, self.base.dose_factor, self.base.I0)
        else:
            ldct = self.base._load_npy(ldct_path)
        if self.mode == 'train':
            h, w = ndct.shape
            ph   = self.base.patch_size
            if h > ph and w > ph:
                top  = np.random.randint(0, h - ph)
                left = np.random.randint(0, w - ph)
                ndct = ndct[top:top+ph, left:left+ph]
                ldct = ldct[top:top+ph, left:left+ph]
        return (torch.from_numpy(ldct).unsqueeze(0).float(),
                torch.from_numpy(ndct).unsqueeze(0).float())


train_set = IndexedMayo(mayo_full, train_pair_indices, mode='train')
val_set   = IndexedMayo(mayo_full, val_pair_indices,   mode='val')
print(f'Train samples: {len(train_set)},  Val samples: {len(val_set)}')

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=CONFIG['num_workers'],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=1,
                          shuffle=False, num_workers=2)


In [ ]:
# === RUN TRAINING ===
model     = REDCNN().to(device)
save_path = os.path.join(CONFIG['output_dir'], 'redcnn_mayo_best.pt')

t0 = time.time()
history = train_model(
    model, train_loader, val_loader,
    epochs=CONFIG['epochs'], lr=CONFIG['lr'], device=device,
    save_path=save_path, val_every=CONFIG['val_every'],
)
print(f'Training time: {(time.time() - t0) / 60:.1f} min')

with open(os.path.join(CONFIG['output_dir'], 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)


## 5. In-Domain Evaluation (Mayo)

Load best checkpoint, evaluate on full Mayo validation set. Compare to LDCT input (no-denoise baseline).


In [ ]:
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt['model'])
print(f"Loaded checkpoint  epoch={ckpt['epoch']}  PSNR={ckpt['psnr']:.2f} dB  SSIM={ckpt['ssim']:.4f}")

# LDCT-input baseline (no denoising)
@torch.no_grad()
def evaluate_input(loader, device):
    psnrs, ssims = [], []
    for ldct, ndct in tqdm(loader, desc='Raw input', leave=False):
        ldct = ldct.to(device); ndct = ndct.to(device)
        psnrs.append(compute_psnr(ldct, ndct))
        ssims.append(compute_ssim(ldct, ndct))
    return float(np.mean(psnrs)), float(np.mean(ssims))

mayo_input_psnr, mayo_input_ssim = evaluate_input(val_loader, device)
mayo_psnr,        mayo_ssim       = evaluate(model, val_loader, device)

print()
print(f'Mayo LDCT input  (no denoise): PSNR={mayo_input_psnr:.2f} dB  SSIM={mayo_input_ssim:.4f}')
print(f'Mayo RED-CNN     (denoised):   PSNR={mayo_psnr:.2f} dB  SSIM={mayo_ssim:.4f}')
print(f'Improvement: +{mayo_psnr - mayo_input_psnr:.2f} dB')

results = {'mayo': {
    'input_psnr':    mayo_input_psnr,  'input_ssim':    mayo_input_ssim,
    'denoised_psnr': mayo_psnr,         'denoised_ssim': mayo_ssim,
    'improvement_db': mayo_psnr - mayo_input_psnr,
}}


## 6. Cross-Vendor Evaluation (Piglet, GE)

**THE KEY EXPERIMENT.** RED-CNN trained on Mayo (Siemens) tested on Piglet (GE). Significant PSNR drop or degraded SSIM/visuals = paper motivation confirmed.


In [ ]:
piglet_test   = PigletDataset(
    CONFIG['piglet_root'], PIGLET_REF, PIGLET_LOW,
    hu_min=CONFIG['hu_min'], hu_max=CONFIG['hu_max'], max_slices=300,
)
piglet_loader = DataLoader(piglet_test, batch_size=1, shuffle=False, num_workers=2)

# Baseline: raw low-mAs vs high-mAs (no denoising)
piglet_input_psnr, piglet_input_ssim = evaluate_input(piglet_loader, device)
print(f'Piglet RAW (no denoise):     PSNR={piglet_input_psnr:.2f} dB  SSIM={piglet_input_ssim:.4f}')

# RED-CNN on Piglet (cross-vendor, no fine-tune)
piglet_psnr, piglet_ssim = evaluate(model, piglet_loader, device)
print(f'Piglet RED-CNN (Mayo-train): PSNR={piglet_psnr:.2f} dB  SSIM={piglet_ssim:.4f}')

results['piglet'] = {
    'ref_series':    PIGLET_REF,
    'low_series':    PIGLET_LOW,
    'input_psnr':    piglet_input_psnr,  'input_ssim':    piglet_input_ssim,
    'denoised_psnr': piglet_psnr,         'denoised_ssim': piglet_ssim,
    'improvement_db': piglet_psnr - piglet_input_psnr,
}


## 7. Results Summary


In [ ]:
with open(os.path.join(CONFIG['output_dir'], 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

# ASCII table
print('=' * 70)
print('WEEK 1 RESULTS')
print('=' * 70)
print(f"{'Setting':<35} {'PSNR (dB)':<12} {'SSIM':<10}")
print('-' * 70)
print(f"{'Mayo LDCT input (no denoise)':<35} {results['mayo']['input_psnr']:<12.2f} {results['mayo']['input_ssim']:<10.4f}")
print(f"{'Mayo RED-CNN (in-domain)':<35} {results['mayo']['denoised_psnr']:<12.2f} {results['mayo']['denoised_ssim']:<10.4f}")
print(f"{'Piglet raw (no denoise)':<35} {results['piglet']['input_psnr']:<12.2f} {results['piglet']['input_ssim']:<10.4f}")
print(f"{'Piglet RED-CNN (cross-vendor)':<35} {results['piglet']['denoised_psnr']:<12.2f} {results['piglet']['denoised_ssim']:<10.4f}")
print('-' * 70)

mayo_gain   = results['mayo']['improvement_db']
piglet_gain = results['piglet']['improvement_db']
gap_psnr    = results['mayo']['denoised_psnr'] - results['piglet']['denoised_psnr']
gap_drop    = mayo_gain - piglet_gain

print(f'Mayo denoising gain:    +{mayo_gain:.2f} dB')
print(f'Piglet denoising gain:  {piglet_gain:+.2f} dB')
print(f'Mayo→Piglet PSNR gap:   {gap_psnr:.2f} dB')
print(f'Gain degradation:       {gap_drop:.2f} dB  (paper-worthy if ≥ 2 dB)')
print('=' * 70)


In [ ]:
# Plots: training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], 'b-')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('L1 loss')
axes[0].set_title('Training loss'); axes[0].grid(alpha=0.3)

if history['val_psnr']:
    ve, vp = zip(*history['val_psnr'])
    axes[1].plot(ve, vp, 'go-')
    axes[1].axhline(35, color='r', linestyle='--', alpha=0.5, label='35 dB target')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR (dB)')
    axes[1].set_title('Mayo validation PSNR'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves.png'), dpi=120)
plt.show()


In [ ]:
# Qualitative: one Piglet slice — low-mAs / RED-CNN denoised / reference
model.eval()
idx = len(piglet_test) // 2
low, ref = piglet_test[idx]
with torch.no_grad():
    denoised = model(low.unsqueeze(0).to(device)).cpu().squeeze()

p_low = compute_psnr(low.unsqueeze(0).to(device), ref.unsqueeze(0).to(device))
p_den = compute_psnr(denoised.unsqueeze(0).unsqueeze(0).to(device),
                     ref.unsqueeze(0).to(device))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(low.squeeze(),       cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Piglet low-mAs ({PIGLET_LOW})\nPSNR vs ref: {p_low:.2f} dB')
axes[1].imshow(denoised,            cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'RED-CNN output (Mayo-trained)\nPSNR vs ref: {p_den:.2f} dB')
axes[2].imshow(ref.squeeze(),       cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Piglet high-mAs reference ({PIGLET_REF})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'piglet_qualitative.png'), dpi=120)
plt.show()


## Week 1 Decision Points

After running this notebook, check:

| Check | Result | Action |
|---|---|---|
| Mayo val PSNR ≥ 35 dB | ✅ | Pipeline OK, proceed to Week 2 |
| Mayo val PSNR 33-35 dB | ⚠️ | Borderline — try longer training or check normalization |
| Mayo val PSNR < 33 dB | ❌ | Pipeline bug — debug before proceeding |
| Piglet gain degradation ≥ 2 dB | ✅ | Cross-vendor story is paper-worthy |
| Piglet gain degradation 1-2 dB | ⚠️ | Weak but acceptable signal |
| Piglet gain degradation < 1 dB | ❌ | Need alternative cross-domain axis |

### What's intentionally simple in Week 1
- Image-domain noise approximation (Zeng sinogram simulation deferred to Week 2)
- Image-level Mayo split (patient-level split for production paper)
- Single Piglet ref/low pair (full dose ladder in Week 2)
- Only RED-CNN baseline (CTformer, CoreDiff in Week 4)

### Next (Week 2)
1. Replace image-domain noise sim with Zeng sinogram-domain (use `torch-radon` or `tomopy`)
2. Add LoRA adapter to RED-CNN (freeze backbone, train rank-16 adapter)
3. Implement noise-injection self-supervision loss: `L = ‖f(Y) − f(Y + extra_poisson)‖₁`
4. Test-time adaptation loop: 5-10 gradient steps on adapter per Piglet slice
5. Re-evaluate Piglet with TTA — target: close cross-vendor gap by ≥ 50%

### Files produced in `output_dir`
- `redcnn_mayo_best.pt` — best Mayo checkpoint (used as starting point for TTA)
- `history.json` — training history
- `results.json` — final PSNR/SSIM table
- `sample_check.png`, `training_curves.png`, `piglet_qualitative.png`
- `piglet_series.json` — Piglet series metadata
